In [ ]:
# !pip install gurobipy

  Using cached gurobipy-13.0.3-cp310-cp310-macosx_10_9_universal2.whl (16.0 MB)

[notice] A new release of pip is available: 23.0.1 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [3]:
import gurobipy as gp
print(f"Gurobi version: {gp.__version__}")

Gurobi version: 13.0.3


[Gurobi tutorial](https://www.datacamp.com/tutorial/gurobi)

### When not to use Gurobi

Gurobi is powerful, but it is not always the right choice:

- **Simple problems:** For small linear programs, free solvers such as `scipy.optimize.linprog` or Excel Solver may be sufficient.

- **Cost:** Commercial licenses can be expensive. For non-critical internal tools, open-source solvers such as CBC or GLPK (via PuLP) may suffice.
- **Real-time decisions:** If a good-enough answer is needed quickly, heuristics such as genetic algorithms may be more suitable than an exact solver.

## Modern Gurobi: The Matrix API


In [4]:
import gurobipy as gp
import numpy as np

# Data
prices = np.array([2.0, 5.0])
resources = np.array([[0.5, 2.0], [1.0, 1.0]])
limits = np.array([100.0, 80.0])

model = gp.Model("bakery_matrix")
x = model.addMVar(shape=2, name="x")

model.setObjective(prices @ x, gp.GRB.MAXIMIZE)
model.addConstr(resources @ x <= limits, name="constraints")

model.optimize()

Restricted license - for non-production use only - expires 2027-11-29
Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (mac64[arm] - Darwin 24.6.0 24G90)

CPU model: Apple M1
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 2 rows, 2 columns and 4 nonzeros (Max)
Model fingerprint: 0xa2c2f031
Model has 2 linear objective coefficients
Coefficient statistics:
  Matrix range     [5e-01, 2e+00]
  Objective range  [2e+00, 5e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [8e+01, 1e+02]

Presolve time: 0.01s
Presolved: 2 rows, 2 columns, 4 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    7.0000000e+30   3.250000e+30   7.000000e+00      0s
       2    2.8000000e+02   0.000000e+00   0.000000e+00      0s

Solved in 2 iterations and 0.01 seconds (0.00 work units)
Optimal objective  2.800000000e+02


## Creating a simple linear programming model

Let's solve a practical problem: a bakery deciding how many loaves of bread and cakes to bake to maximize profit.

### Problem setup

| Product | Profit per item | Flour per item |
| --- | ---: | ---: |
| Bread | $2 | 0.5 kg |
| Cake | $5 | 2 kg |

- **Flour available:** 100 kg
- **Oven capacity:** 80 items

In [5]:
import gurobipy as gp
from gurobipy import GRB

model = gp.Model("bakery_optimization")

bread = model.addVar(name="bread", vtype=GRB.CONTINUOUS, lb=0)
cake = model.addVar(name="cake", vtype=GRB.CONTINUOUS, lb=0)

model.setObjective(2 * bread + 5 * cake, GRB.MAXIMIZE)

model.addConstr(0.5 * bread + 2 * cake <= 100, name="flour_limit")
model.addConstr(bread + cake <= 80, name="oven_capacity")

model.optimize()

if model.status == GRB.OPTIMAL:
    print(f"Optimal profit: ${model.objVal:.2f}")
    print(f"Bake {bread.X:.2f} loaves of bread")
    print(f"Bake {cake.X:.2f} cakes")
else:
    print("No optimal solution found")

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (mac64[arm] - Darwin 24.6.0 24G90)

CPU model: Apple M1
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 2 rows, 2 columns and 4 nonzeros (Max)
Model fingerprint: 0xa2c2f031
Model has 2 linear objective coefficients
Coefficient statistics:
  Matrix range     [5e-01, 2e+00]
  Objective range  [2e+00, 5e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [8e+01, 1e+02]

Presolve time: 0.00s
Presolved: 2 rows, 2 columns, 4 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    7.0000000e+30   3.250000e+30   7.000000e+00      0s
       2    2.8000000e+02   0.000000e+00   0.000000e+00      0s

Solved in 2 iterations and 0.01 seconds (0.00 work units)
Optimal objective  2.800000000e+02
Optimal profit: $280.00
Bake 40.00 loaves of bread
Bake 40.00 cakes


In [1]:
from ortools.sat.python import cp_model

# CP-SAT uses integer variables; this models whole loaves and cakes.
cp_model_bakery = cp_model.CpModel()
cp_bread = cp_model_bakery.new_int_var(0, 80, "bread")
cp_cake = cp_model_bakery.new_int_var(0, 80, "cake")

# Multiply the flour constraint by 2 to use integer coefficients.
cp_model_bakery.add(cp_bread + 4 * cp_cake <= 200)
cp_model_bakery.add(cp_bread + cp_cake <= 80)
cp_model_bakery.maximize(2 * cp_bread + 5 * cp_cake)

cp_solver = cp_model.CpSolver()
cp_status = cp_solver.solve(cp_model_bakery)

if cp_status == cp_model.OPTIMAL:
    print(f"Optimal profit: ${cp_solver.objective_value:.2f}")
    print(f"Bake {cp_solver.value(cp_bread)} loaves of bread")
    print(f"Bake {cp_solver.value(cp_cake)} cakes")
elif cp_status == cp_model.FEASIBLE:
    print(f"Feasible profit (not proven optimal): ${cp_solver.objective_value:.2f}")
    print(f"Bake {cp_solver.value(cp_bread)} loaves of bread")
    print(f"Bake {cp_solver.value(cp_cake)} cakes")
else:
    print(f"No solution found. Solver status: {cp_solver.status_name(cp_status)}")

Optimal profit: $280.00
Bake 40 loaves of bread
Bake 40 cakes


## Advanced Problem Types Supported by Gurobi

Linear programming is just the start. Gurobi handles complex types that capture real-world nuances, including the new Non-Convex MINLP capabilities in version 13.0.

### Mixed-integer linear programming

#### What makes it different

Mixed-integer programming (MIP) allows variables to be integers. You can't build 0.7 factories, so you need integer solutions.

#### Real-world applications

- **Facility location:** Open a warehouse in City A? Yes (1) or No (0).
- **Production scheduling:** Batch sizes often must be whole numbers.
- **Network design:** Routing decisions are usually binary. Traffic flows through a link or it doesn't.

In [2]:
import gurobipy as gp
from gurobipy import GRB

model = gp.Model("facility_location")

cities = ['CityA', 'CityB', 'CityC']
fixed_costs = {'CityA': 10000, 'CityB': 8000, 'CityC': 12000}

open_warehouse = model.addVars(cities, vtype=GRB.BINARY, name="open")

model.setObjective(
    gp.quicksum(fixed_costs[city] * open_warehouse[city] for city in cities),
    GRB.MINIMIZE
)

model.addConstr(gp.quicksum(open_warehouse[city] for city in cities) >= 2)

model.optimize()

print("Open warehouses in:")
for city in cities:
    if open_warehouse[city].X > 0.5:
        print(f"  {city}")

Restricted license - for non-production use only - expires 2027-11-29
Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (mac64[arm] - Darwin 24.6.0 24G90)

CPU model: Apple M1
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 1 rows, 3 columns and 3 nonzeros (Min)
Model fingerprint: 0x39ba1405
Model has 3 linear objective coefficients
Variable types: 0 continuous, 3 integer (3 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [8e+03, 1e+04]
  Bounds range     [1e+00, 1e+00]
  RHS range        [2e+00, 2e+00]

Found heuristic solution: objective 20000.000000
Presolve removed 1 rows and 3 columns
Presolve time: 0.00s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.01 seconds (0.00 work units)
Thread count was 1 (of 8 available processors)

Solution count 2: 18000 20000 

Optimal solution found (tolerance 1.00e-04)
Best objective 1.800000000000e+04, best bound 1.800000000

In [3]:
from ortools.sat.python import cp_model

cp_cities = ['CityA', 'CityB', 'CityC']
cp_fixed_costs = {'CityA': 10000, 'CityB': 8000, 'CityC': 12000}

cp_facility_model = cp_model.CpModel()
cp_open_warehouse = {
    city: cp_facility_model.new_bool_var(f'open_{city}') for city in cp_cities
}
cp_facility_model.add(sum(cp_open_warehouse.values()) >= 2)
cp_facility_model.minimize(
    sum(cp_fixed_costs[city] * cp_open_warehouse[city] for city in cp_cities)
)

cp_facility_solver = cp_model.CpSolver()
cp_facility_status = cp_facility_solver.solve(cp_facility_model)

if cp_facility_status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    label = 'Optimal' if cp_facility_status == cp_model.OPTIMAL else 'Feasible (not proven optimal)'
    print(f'{label} cost: ${cp_facility_solver.objective_value:,.2f}')
    print('Open warehouses in:')
    for city in cp_cities:
        if cp_facility_solver.value(cp_open_warehouse[city]):
            print(f'  {city}')
else:
    print(f'No solution found. Solver status: {cp_facility_solver.status_name(cp_facility_status)}')

Optimal cost: $18,000.00
Open warehouses in:
  CityA
  CityB
